# Using machine learning to identify former farmland based on current characteristics

- Author: Dillon Haller
- Date: December 1, 2025
- Class: CMSE 802
- [Link to GitHub Repo](https://github.com/DillonJHaller/cmse802_project.git)

### Background and Motivation
Land that was formerly under cultivation may have distinct soil, vegetation, etc. characteristics from that which was not recently under cultivation. For instance, eastern red-cedar (<i>Juniperus virginiana</i>) is a pioneer species on the North American Great Plains which often expands quickly in open fields when not suppressed by fire or human activity. The different species assemblages present in former farmland may make it distinct in a way that creates detectable differences in the satellite record. If this is the case, it could allow investigation into past land cover and land use earlier than when widespread satellite observations became available, simply by studying the characteristics of that land that are visible in the historical satellite data that is available. It could also allow for more robust validation of long-term land cover classification datasets like the National Land Cover Dataset. 
I focus my efforts on central Missouri, an area which I know well, and which has a lot of land that frequently shifts between cropping, pasture, and fallow periods. My study area covers four Sentinel-2 tiles, with an approximate total area of 45000 square kilometers. I used the following datasets:
- National Land Cover Dataset
- Harmonized Landsat / Sentinel-2

(See more under `Data_Statement.md`)

My research questions were:
- Can I use remote sensing data and machine learning techniques to detect land that was formerly in agricultural use based on more recent observations?
- What land cover characteristics distinguish former agricultural land from land that has not been in recent agricultural use?


### Methodology
The methodology for this project broke down into four essential steps:

#### NLCD processing
In order to generate labels for the machine learning models, I needed to compute land cover trajectories for all of the study area. I focused on the past ten years to avoid having too many complicated transitions. Labels take the form of "Long-term pattern classes" (LTPCs), a concept used by Martin et al., (2025) to identify former farmland. I have expanded on this to identify all transitions between cropland, pasture, and "Non-agricultural, non-developed" (NAND) land, which creates a total of nine LTPCs of interest. These are labeled as follows:

```
        Constants
            1: Stable pasture/hay
            2: Stable cropland
            3: Stable non-agricultural/non-developed
        Transitions
            4: Transitioned from pasture/hay to cropland
            5: Transitioned from pasture to non-agricultural/non-developed
            6: Transitioned from cropland to pasture
            7: Transitioned from cropland to NAND
            8: Transitioned from NAND to pasture
            9: Transitioned from NAND to crops
        Other
            0: NoData or erratic patterns 
            11: Developed land at any time
```

with classes 0 and 11 being left out of further analysis

#### HLS processing
Feature data for this project came from the Harmonized Landsat / Sentinel-2 (HLS) dataset, which records surface reflectance values in a variety of different bands across the visible and infrared portions of the electromagnetic spectrum. Raw values are too numerous to be used as feature values alone, so for each band, I computed the average value across the whole year, as well as just the winter and just the summer. I then also computed the EVI2 index of Jiang et al., (2008), which combines red and infrared reflectance data into an index that is often used as a proxy for vegetation density, for the same time periods.

#### Training and testing data generation
I randomly selected 100 points for training and 25 points for testing from each LTPC to generate my dataset. Selecting just a sub-sample of the full data both saves on computation time and also neatly avoids the issue of imbalance in the frequency of the different LTPCs. Those points were exported to 2 shapefiles, keeping the training and testing datasets separate from the very beginning. I then pulled data from each seasonal and yearly band average, as well as the EVI2 index, with each band or index serving as a feature to the output dataset. As a final data generation task, I did some feature engineering by combining very similar bands from Landsat and Sentinel-2 into single features to address potential overfitting, and then also computed the difference between summer and winter values for each band or index.

#### Model implementation
Because the data generation was quite intensive, I chose to keep the modelling fairly simple. I used grid searches with 5-fold cross validation to train both a random forest and a support vector machine model. I then ran these models on the testing dataset to generate confusion matrices for final evaluation. In order to answer the question of what features actually distinguished the LTPCs from one another, I then performed some rudimentary SHAPly analysis as a final step for the project.


### Results
The random forest model and the support vector model had very similar performances, achieving best cross-validation accuracies of 41.9% and 42.8%, respectively, in the final model. True random chance accuracy would be about 11%, given that there are 9 total LTPCs, so both models do perform better than chance. However, given that we can already discriminate between end-state land cover classes, I would say that the true floor for being able to identify the historical land cover would be a 33% accuracy. These models performed slightly better than that benchmark, suggesting that the answer to the first research question is a weak yes.

Confusion matrix analysis (see below) reveals that the models performed much better with some LTPCs than others, with class-specific producer's accuracy values ranging between 76% and 16%. Unsurprisingly, they tended to perform a bit better with the constant classes than the change classes. Also unsurprisingly, confusion was most common between LTPCs with the same end-state land cover. The random forest model was a bit more accurate at predicting crop -> NAND transition classes, but otherwise the models were again similar.

![c1](results/figures/random_forest_confusion_matrix.png)
![c2](results/figures/support_vector_confusion_matrix.png)

Brief analysis of the most important factors in the model was the most surprising result. Shortwave infrared values tended to be the most important, overshadowing the EVI2 index as well as the bands that went into it. I was also surprised to see that yearly values tended to be more important than the difference values.

![rf](results/figures/Model_analysis/rf_feature_importances.png)
![sv](results/figures/Model_analysis/svm_shap_summary.png)

### Discussion and Reflection

Overall performance of the models was somewhat disappointing, but not unexpectedly so. I expect that no model will be able to achieve tremendously good accuracy here, since every patch of landscape is different. The most surprising results for me came toward the end, when looking at feature importances and SHAPly values. The features which I had expected to be most important were the summer/winter difference values and the EVI2 values. Instead, the models relied quite heavily on yearly averages and at least a little on shortwave infrared, which does not often figure heavily in vegetation tracking. This shows the importance of keeping in variables that could contain useful information, and the power of machine learning to pick up on unexpected patterns.

While working on this project, I found it was difficult to get through the necessary pre-processing scripts to actually get to the machine learning, and did not get as far as I would have liked on analyzing feature importances. With these developed scripts now in hand, I hope to be able to continue this work in the future. 

I already have some future steps in mind:
- I plan to continue feature engineering, which is essentially the largest part of any remote sensing-based machine learning project. I will add vegetation seasonality indicators which can easily track earlier and later green-ups, distinguishing better between different vegetation assemblages
- I plan to bring in more years of data and compute trajectories of these values over a period of five years or so, which may provide more ability to distinguish the constant LTPCs from the changing LTPCs
- I expect model overfitting may still be an issue, given that reflectance values from similar bands are often highly correlated, although there is not much evidence of it in the results here. I still expect to potentially need to drop problematic features after more runs of the model.
- Time permitting, I may also bring in radar remote sensing data from Sentinel-1, which can provide a unique extra dimension not captured by passive, visible and infrared remote sensing.